# **Phần 2: Phân rã ma trận và Chéo hóa (Demo)**

**Thông tin nhóm:**
* **Môn học:** Toán Ứng Dụng và Thống Kê
* **Trường:** ĐH Khoa học Tự nhiên - ĐHQG TP.HCM
* **Nhóm:** 8
* **Lớp:** CQ2024/2
* **Thành viên:**
  1.   Nguyễn Đình Tuấn - 24120237
  2.   Nguyễn Anh Thái - 24120224
  3.   Nguyễn Huỳnh Gia Bảo - 24120264
  4.   Vòng Sau Hậu - 24120307
  5.   Lương Nhật Tân - 24120134

---

## **Giới thiệu tổng quan**
Notebook này demo việc kiểm thử các thuật toán bằng Python, bao gồm:
1. Kiểm thử thuật toán phân rã PA = LU  (LU Decomposition)
2. Kiểm thử thuật toán phân rã QR (QR Decomposition)
3. Kiểm thử thuật toán phân rã SVD (Singular Value Decomposition)
4. Kiểm thử thuật toán chéo hóa ma trận (Diagonalization)
...

# **1. Thuật toán Phân rã PA = LU (Tự lập trình với SciPy)**

Dùng `numpy` để nhân ma trận kiểm tra ($P \cdot A = L \cdot U$) và dùng `scipyV.linalg.lu` để đối chiếu kết quả.

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from decomposition import LUDecomposition
import numpy as np
from scipy.linalg import lu

## 1.1. **Định nghĩa hàm kiểm thử**

In [13]:
def test_lu_decomposition(matrix_name, A_list):
    print(f"KIỂM THỬ: {matrix_name}")

    # Chạy thuật toán tự viết
    solver = LUDecomposition(A_list)
    P_custom, L_custom, U_custom = solver.decompose()

    # Chuyển đổi sang numpy array
    A = np.array(A_list, dtype=float)
    P = np.array(P_custom)
    L = np.array(L_custom)
    U = np.array(U_custom)

    # Kiểm tra: P * A == L * U
    PA = P @ A
    LU = L @ U
    is_pa_lu_equal = np.allclose(PA, LU)

    # So sánh với SciPy
    P_scipy, L_scipy, U_scipy = lu(A)
    # scipy trả về P thỏa mãn A = P @ L @ U nên phải transpose P để thành P.T @ A = L @ U
    P_scipy_transposed = P_scipy.T

    print("1. Kiểm tra P * A = L * U (Thuật toán tự viết):", "ĐÚNG" if is_pa_lu_equal else "SAI")
    if not is_pa_lu_equal:
        print("  - PA:\n", np.round(PA, 4))
        print("  - LU:\n", np.round(LU, 4))

    print("2. So sánh hình dạng ma trận (Tự code vs SciPy):")
    print(f"  - P: Custom {P.shape} | SciPy {P_scipy_transposed.shape}")
    print(f"  - L: Custom {L.shape} | SciPy {L_scipy.shape}")
    print(f"  - U: Custom {U.shape} | SciPy {U_scipy.shape}")

    print("3. So sánh kết quả chi tiết với SciPy:")
    if P.shape == P_scipy_transposed.shape:
        print("   - P:", "KHỚP" if np.allclose(P, P_scipy_transposed) else "KHÁC NHAU (Bình thường do khác quy tắc chọn Pivot)")
    else:
        print("   - P: KHÔNG THỂ SO SÁNH (Khác kích thước)")

    if L.shape == L_scipy.shape:
        print("   - L:", "KHỚP" if np.allclose(L, L_scipy) else "KHÁC NHAU")
    else:
        print("   - L: KHÔNG THỂ SO SÁNH BẰNG LỆNH (Vì SciPy dùng dạng rút gọn)")

    if U.shape == U_scipy.shape:
        print("   - U:", "KHỚP" if np.allclose(U, U_scipy) else "KHÁC NHAU")
    else:
        print("   - U: KHÔNG THỂ SO SÁNH BẰNG LỆNH (Vì SciPy dùng dạng rút gọn)")

    print("\nMa trận P (Tự code):\n", np.round(P, 4))
    print("\nMa trận L (Tự code):\n", np.round(L, 4))
    print("\nMa trận U (Tự code):\n", np.round(U, 4))

## 1.2. **Test Case 1: Ma trận vuông 3x3**

In [14]:
A_square = [
    [2, 1, 1],
    [4, -6, 0],
    [-2, 7, 2]
]
test_lu_decomposition("MA TRẬN VUÔNG (3x3)", A_square)

KIỂM THỬ: MA TRẬN VUÔNG (3x3)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (3, 3) | SciPy (3, 3)
  - L: Custom (3, 3) | SciPy (3, 3)
  - U: Custom (3, 3) | SciPy (3, 3)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHỚP
   - U: KHỚP

Ma trận P (Tự code):
 [[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]

Ma trận L (Tự code):
 [[ 1.   0.   0. ]
 [ 0.5  1.   0. ]
 [-0.5  1.   1. ]]

Ma trận U (Tự code):
 [[ 4. -6.  0.]
 [ 0.  4.  1.]
 [ 0.  0.  1.]]


## 1.3. **Test Case 2: Ma trận chữ nhật (4x3) - Nhiều hàng hơn cột**

In [15]:
A_rect_4x3 = [
    [1, 2, 3],
    [2, -4, 6],
    [3, -9, -3],
    [-2, 4, -1]
]
test_lu_decomposition("MA TRẬN CHỮ NHẬT (4x3)", A_rect_4x3)

KIỂM THỬ: MA TRẬN CHỮ NHẬT (4x3)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (4, 4) | SciPy (4, 4)
  - L: Custom (4, 4) | SciPy (4, 3)
  - U: Custom (4, 3) | SciPy (3, 3)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHÔNG THỂ SO SÁNH BẰNG LỆNH (Vì SciPy dùng dạng rút gọn)
   - U: KHÔNG THỂ SO SÁNH BẰNG LỆNH (Vì SciPy dùng dạng rút gọn)

Ma trận P (Tự code):
 [[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]

Ma trận L (Tự code):
 [[ 1.      0.      0.      0.    ]
 [ 0.3333  1.      0.      0.    ]
 [ 0.6667  0.4     1.      0.    ]
 [-0.6667 -0.4    -0.2187  1.    ]]

Ma trận U (Tự code):
 [[ 3.  -9.  -3. ]
 [ 0.   5.   4. ]
 [ 0.   0.   6.4]
 [ 0.   0.   0. ]]


## 1.4. **Test Case 3: Ma trận chữ nhật (2x4) - Nhiều cột hơn hàng**

In [16]:
A_rect_2x4 = [
    [2, 4, -1, 5],
    [-4, -5, 3, -8]
]
test_lu_decomposition("MA TRẬN CHỮ NHẬT (2x4)", A_rect_2x4)

KIỂM THỬ: MA TRẬN CHỮ NHẬT (2x4)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (2, 2) | SciPy (2, 2)
  - L: Custom (2, 2) | SciPy (2, 2)
  - U: Custom (2, 4) | SciPy (2, 4)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHỚP
   - U: KHỚP

Ma trận P (Tự code):
 [[0. 1.]
 [1. 0.]]

Ma trận L (Tự code):
 [[ 1.   0. ]
 [-0.5  1. ]]

Ma trận U (Tự code):
 [[-4.  -5.   3.  -8. ]
 [ 0.   1.5  0.5  1. ]]


## 1.5. **Test Case 4: Ma trận chứa cột toàn số 0**

In [17]:
A_singular = [
    [0, 2, 3],
    [0, 4, 6],
    [0, 6, 9]
]
test_lu_decomposition("MA TRẬN SUY BIẾN / CỘT BẰNG 0 (3x3)", A_singular)

KIỂM THỬ: MA TRẬN SUY BIẾN / CỘT BẰNG 0 (3x3)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (3, 3) | SciPy (3, 3)
  - L: Custom (3, 3) | SciPy (3, 3)
  - U: Custom (3, 3) | SciPy (3, 3)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHỚP
   - U: KHỚP

Ma trận P (Tự code):
 [[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]

Ma trận L (Tự code):
 [[1.     0.     0.    ]
 [0.     1.     0.    ]
 [0.     0.6667 1.    ]]

Ma trận U (Tự code):
 [[0. 2. 3.]
 [0. 6. 9.]
 [0. 0. 0.]]


## **1.6. Test Case 5: Ma trận đã ở sẵn dạng tam giác trên (Upper Triangular)**
Mục đích: Kiểm tra xem thuật toán có đủ thông minh để nhận diện ma trận đã chuẩn sẵn hay không. Nếu code chuẩn, P và L sẽ tự động là ma trận đơn vị, U = A.

In [18]:
A_upper = [
    [2.0, 3.0,  4.0],
    [0.0, 5.0, -1.0],
    [0.0, 0.0,  7.0]
]
test_lu_decomposition("EDGE CASE 3: MA TRẬN TAM GIÁC TRÊN (Đã chuẩn sẵn)", A_upper)

KIỂM THỬ: EDGE CASE 3: MA TRẬN TAM GIÁC TRÊN (Đã chuẩn sẵn)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (3, 3) | SciPy (3, 3)
  - L: Custom (3, 3) | SciPy (3, 3)
  - U: Custom (3, 3) | SciPy (3, 3)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHỚP
   - U: KHỚP

Ma trận P (Tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Ma trận L (Tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Ma trận U (Tự code):
 [[ 2.  3.  4.]
 [ 0.  5. -1.]
 [ 0.  0.  7.]]


## 1.7. **Test Case 6: Ma trận đường chéo phụ**
Mục đích: Thuật toán bắt buộc phải thực hiện hoán vị (swap) ở tất cả các bước để kéo các số 0 xuống dưới. Kiểm tra khả năng xử lý ma trận P và hoán đổi L.

In [19]:
A_anti_diag = [
    [0.0, 0.0, 3.0],
    [0.0, 2.0, 0.0],
    [1.0, 0.0, 0.0]
]
test_lu_decomposition("EDGE CASE 4: MA TRẬN ĐƯỜNG CHÉO PHỤ (Hoán vị liên tục)", A_anti_diag)

KIỂM THỬ: EDGE CASE 4: MA TRẬN ĐƯỜNG CHÉO PHỤ (Hoán vị liên tục)
1. Kiểm tra P * A = L * U (Thuật toán tự viết): ĐÚNG
2. So sánh hình dạng ma trận (Tự code vs SciPy):
  - P: Custom (3, 3) | SciPy (3, 3)
  - L: Custom (3, 3) | SciPy (3, 3)
  - U: Custom (3, 3) | SciPy (3, 3)
3. So sánh kết quả chi tiết với SciPy:
   - P: KHỚP
   - L: KHỚP
   - U: KHỚP

Ma trận P (Tự code):
 [[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]

Ma trận L (Tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Ma trận U (Tự code):
 [[1. 0. 0.]
 [0. 2. 0.]
 [0. 0. 3.]]


# **2. Thuật toán QR Decomposition**

In [20]:
from decomposition import svd, transpose, multiply,qr_decomposition

## 2.1. **Test case 1: Ma trận vuông 3x3**

In [21]:
A1 = [[12, -51, 4],
      [6, 167, -68],
      [-4, 24, -41]]

Q1, R1 = qr_decomposition(A1)

print("====TEST 1: Ma trận vuông 3x3====")
print("Ma trận Q:")
for row in Q1: print([round(x, 4) for x in row])

print("\nMa trận R (Tam giác trên):")
for row in R1: print([round(x, 4) for x in row])

A1_rec = multiply(Q1, R1)
print("\nMa trận tái tạo (Q * R):")
for row in A1_rec: print([round(x, 4) for x in row])

====TEST 1: Ma trận vuông 3x3====
Ma trận Q:
[0.8571, -0.3943, -0.3314]
[0.4286, 0.9029, 0.0343]
[-0.2857, 0.1714, -0.9429]

Ma trận R (Tam giác trên):
[14.0, 21.0, -14.0]
[0.0, 175.0, -70.0]
[0.0, 0.0, 35.0]

Ma trận tái tạo (Q * R):
[12.0, -51.0, 4.0]
[6.0, 167.0, -68.0]
[-4.0, 24.0, -41.0]


## 2.2. **Test case 2: Ma trận 4x2**

In [22]:
A2 = [[1, 2],
      [3, 4],
      [5, 6],
      [7, 8]]

Q2, R2 = qr_decomposition(A2)

print("====TEST 2: Ma trận chữ nhật 4x2====")
print("Ma trận Q:")
for row in Q2: print([round(x, 4) for x in row])

print("\nMa trận R:")
for row in R2: print([round(x, 4) for x in row])

A2_rec = multiply(Q2, R2)
print("\nMa trận tái tạo (Q * R):")
for row in A2_rec: print([round(x, 4) for x in row])

====TEST 2: Ma trận chữ nhật 4x2====
Ma trận Q:
[0.1091, 0.8295]
[0.3273, 0.4392]
[0.5455, 0.0488]
[0.7638, -0.3416]

Ma trận R:
[9.1652, 10.9109]
[0.0, 0.9759]

Ma trận tái tạo (Q * R):
[1.0, 2.0]
[3.0, 4.0]
[5.0, 6.0]
[7.0, 8.0]


## 2.3. **Test case 3: Ma trận suy biến**

In [23]:
A3 = [[1, 1, 5],
      [2, 2, 6],
      [3, 3, 7]]

Q3, R3 = qr_decomposition(A3)

print("====TEST 3: Ma trận suy biến====")
print("Ma trận Q:")
for row in Q3: print([round(x, 4) for x in row])

print("\nMa trận R:")
for row in R3: print([round(x, 4) for x in row])

A3_rec = multiply(Q3, R3)
print("\nMa trận tái tạo (Q * R):")
for row in A3_rec: print([round(x, 4) for x in row])

====TEST 3: Ma trận suy biến====
Ma trận Q:
[0.2673, 0.0, 0.8729]
[0.5345, 0.0, 0.2182]
[0.8018, 0.0, -0.4364]

Ma trận R:
[3.7417, 3.7417, 10.1559]
[0.0, 0.0, 0.0]
[0.0, 0.0, 2.6186]

Ma trận tái tạo (Q * R):
[1.0, 1.0, 5.0]
[2.0, 2.0, 6.0]
[3.0, 3.0, 7.0]


## 2.4. **Test case 4: Ma trận đơn vị 3x3**

In [24]:
A4 = [[1, 0, 0],
      [0, 1, 0],
      [0, 0, 1]]

Q4, R4 = qr_decomposition(A4)

print("====TEST 4: Ma trận đơn vị 3x3====")
print("Ma trận Q (Phải là ma trận đơn vị):")
for row in Q4: print([round(x, 4) for x in row])

print("\nMa trận R (Phải là ma trận đơn vị):")
for row in R4: print([round(x, 4) for x in row])

A4_rec = multiply(Q4, R4)
print("\nMa trận tái tạo (Q * R):")
for row in A4_rec: print([round(x, 4) for x in row])

====TEST 4: Ma trận đơn vị 3x3====
Ma trận Q (Phải là ma trận đơn vị):
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]

Ma trận R (Phải là ma trận đơn vị):
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]

Ma trận tái tạo (Q * R):
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]


## 2.5. **Test case 5: Ma trận 4x4**

In [25]:
A5 = [[4, 1, 3, 2],
      [1, 5, 0, 1],
      [3, 0, 6, 2],
      [2, 1, 2, 7]]

Q5, R5 = qr_decomposition(A5)

print("====TEST 5: Ma trận vuông 4x4====")
print("Ma trận Q:")
for row in Q5: print([round(x, 4) for x in row])

print("\nMa trận R:")
for row in R5: print([round(x, 4) for x in row])

A5_rec = multiply(Q5, R5)
print("\nMa trận tái tạo (Q * R):")
for row in A5_rec: print([round(x, 4) for x in row])

====TEST 5: Ma trận vuông 4x4====
Ma trận Q:
[0.7303, -0.0974, -0.5943, -0.3225]
[0.1826, 0.9668, 0.1316, -0.1209]
[0.5477, -0.2295, 0.7909, -0.1478]
[0.3651, 0.0556, -0.0635, 0.9271]

Ma trận R:
[5.4772, 2.0083, 6.2075, 5.2947]
[0.0, 4.7924, -1.558, 0.7025]
[0.0, 0.0, 2.8353, 0.0804]
[0.0, 0.0, 0.0, 5.4283]

Ma trận tái tạo (Q * R):
[4.0, 1.0, 3.0, 2.0]
[1.0, 5.0, 0.0, 1.0]
[3.0, 0.0, 6.0, 2.0]
[2.0, 1.0, 2.0, 7.0]


# **3. Thuật toán SVD (Singular Value Decomposition)**

## 3.1. **Test case 1: Ma trận vuông đơn giản 2x2**

In [26]:
A1 = [[1, 2], [3, 4]]
U1, S1, V1 = svd(A1)

print("==== TEST SVD: A1 ====")
print("Ma trận U:")
for row in U1: print([round(x, 4) for x in row])

print("\nMa trận S:")
for row in S1: print([round(x, 4) for x in row])

print("\nMa trận V:")
for row in V1: print([round(x, 4) for x in row])

# Kiểm tra lại: A = U * S * V^T
A1_rec = multiply(multiply(U1, S1), transpose(V1))
print("\nMa trận tái tạo (U * S * V^T):")
for row in A1_rec: print([round(x, 4) for x in row])

==== TEST SVD: A1 ====
Ma trận U:
[0.4046, 0.9145]
[0.9145, -0.4046]

Ma trận S:
[5.465, 0.0]
[0.0, 0.366]

Ma trận V:
[0.576, -0.8174]
[0.8174, 0.576]

Ma trận tái tạo (U * S * V^T):
[1.0, 2.0]
[3.0, 4.0]


## 3.2. **Test case 2: Ma trận suy biến**

In [27]:
A2 = [[1, 2], [2, 4]]
U2, S2, V2 = svd(A2)

print("====TEST 2: Ma trận suy biến====")
print("Ma trận U:")
for row in U2: print([round(x, 4) for x in row])

print("\nMa trận S (Phải có 1 giá trị là 5.0 và 1 giá trị là 0):")
for row in S2: print([round(x, 4) for x in row])

print("\nMa trận V:")
for row in V2: print([round(x, 4) for x in row])

A2_rec = multiply(multiply(U2, S2), transpose(V2))
print("\nMa trận tái tạo (U*S*V^T):")
for row in A2_rec: print([round(x, 4) for x in row])

====TEST 2: Ma trận suy biến====
Ma trận U:
[0.4472, 0.8944]
[0.8944, -0.4472]

Ma trận S (Phải có 1 giá trị là 5.0 và 1 giá trị là 0):
[5.0, 0.0]
[0.0, 0.0]

Ma trận V:
[0.4472, 0.8944]
[0.8944, -0.4472]

Ma trận tái tạo (U*S*V^T):
[1.0, 2.0]
[2.0, 4.0]


## 3.3. **Test case 3: Ma trận trực giao**

In [28]:
A3 = [[0, -1],
      [1, 0]]
U3, S3, V3 = svd(A3)

print("====TEST 3: Ma trận trực giao====")
print("Ma trận U:")
for row in U3: print([round(x, 4) for x in row])

print("\nMa trận S:")
for row in S3: print([round(x, 4) for x in row])

print("\nMa trận V:")
for row in V3: print([round(x, 4) for x in row])

A3_rec = multiply(multiply(U3, S3), transpose(V3))
print("\nMa trận tái tạo (U*S*V^T):")
for row in A3_rec: print([round(x, 4) for x in row])

====TEST 3: Ma trận trực giao====
Ma trận U:
[0.0, -1.0]
[1.0, 0.0]

Ma trận S:
[1.0, 0.0]
[0.0, 1.0]

Ma trận V:
[1.0, 0.0]
[0.0, 1.0]

Ma trận tái tạo (U*S*V^T):
[0.0, -1.0]
[1.0, 0.0]


## 3.4. **Test case 4: Ma trận vector cột (3x1)**

In [29]:
A4 = [[3],
      [4],
      [0]]
U4, S4, V4 = svd(A4)

print("====TEST 4: Ma trận vector cột====")
print("Ma trận U:")
for row in U4: print([round(x, 4) for x in row])

print("\nMa trận S (Giá trị duy nhất phải là 5.0):")
for row in S4: print([round(x, 4) for x in row])

print("\nMa trận V:")
for row in V4: print([round(x, 4) for x in row])

A4_rec = multiply(multiply(U4, S4), transpose(V4))
print("\nMa trận tái tạo (U*S*V^T):")
for row in A4_rec: print([round(x, 4) for x in row])

====TEST 4: Ma trận vector cột====
Ma trận U:
[0.6, 0.8, 0.0]
[0.8, -0.6, 0.0]
[0.0, 0.0, 1.0]

Ma trận S (Giá trị duy nhất phải là 5.0):
[5.0]
[0.0]
[0.0]

Ma trận V:
[1.0]

Ma trận tái tạo (U*S*V^T):
[3.0]
[4.0]
[0.0]


## 3.5. **Test case 5: Ma trận toàn giá trị 0**

In [30]:
A5 = [[0, 0],
      [0, 0]]

print("====TEST 5: Ma trận zero====")
try:
    U5, S5, V5 = svd(A5)
    print("Ma trận U:")
    for row in U5: print([round(x, 4) for x in row])

    print("\nMa trận S:")
    for row in S5: print([round(x, 4) for x in row])

    A5_rec = multiply(multiply(U5, S5), transpose(V5))
    print("\nMa trận tái tạo:")
    for row in A5_rec: print([round(x, 4) for x in row])
except Exception as e:
    print("\nLỗi thuật toán khi gặp ma trận 0:", e)

====TEST 5: Ma trận zero====
Ma trận U:
[1.0, 0.0]
[0.0, 1.0]

Ma trận S:
[0.0, 0.0]
[0.0, 0.0]

Ma trận tái tạo:
[0.0, 0.0]
[0.0, 0.0]


## 3.6. **Test case 6: Ma trận cỡ 4x4**

In [31]:
A6 = [[4, 1, 3, 2],
      [1, 5, 0, 1],
      [3, 0, 6, 2],
      [2, 1, 2, 7]]

U6, S6, V6 = svd(A6)

print("====TEST 6: Ma trận vuông 4x4====")
print("Ma trận U:")
for row in U6: print([round(x, 4) for x in row])

print("\nMa trận S (Phải có 4 giá trị suy biến trên đường chéo):")
for row in S6: print([round(x, 4) for x in row])

print("\nMa trận V:")
for row in V6: print([round(x, 4) for x in row])

A6_rec = multiply(multiply(U6, S6), transpose(V6))

print("\nMa trận tái tạo (U*S*V^T):")
for row in A6_rec: print([round(x, 4) for x in row])

====TEST 6: Ma trận vuông 4x4====
Ma trận U:
[0.479, -0.1275, -0.297, -0.8161]
[0.1959, 0.7347, -0.6103, 0.2223]
[0.5784, -0.5513, -0.2851, 0.5294]
[0.6306, 0.3744, 0.6767, 0.0653]

Ma trận S (Phải có 4 giá trị suy biến trên đường chéo):
[10.6645, 0.0, 0.0, 0.0]
[0.0, 5.3359, 0.0, 0.0]
[0.0, 0.0, 4.3778, 0.0]
[0.0, 0.0, 0.0, 1.6217]

Ma trận V:
[0.479, -0.1275, -0.297, -0.8161]
[0.1959, 0.7347, -0.6103, 0.2223]
[0.5784, -0.5513, -0.2851, 0.5294]
[0.6306, 0.3744, 0.6767, 0.0653]

Ma trận tái tạo (U*S*V^T):
[4.0, 1.0, 3.0, 2.0]
[1.0, 5.0, 0.0, 1.0]
[3.0, 0.0, 6.0, 2.0]
[2.0, 1.0, 2.0, 7.0]


# **4. Thuật toán Chéo Hóa Ma Trận A = P·D·P⁻¹ (Tự lập trình với NumPy)**

Dùng `numpy` để kiểm tra đẳng thức $A = P \cdot D \cdot P^{-1}$ và $P^{-1} \cdot A \cdot P = D$, đồng thời đối chiếu với `numpy.linalg.eig` để xác nhận kết quả.

In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
from diagonalization import diagonalize
import numpy as np

## 4.1. **Định nghĩa hàm kiểm thử**

In [34]:
def test_diagonalization(matrix_name, A_list, expect_none=False):
    print(f"KIỂM THỬ: {matrix_name}")
    print("-" * 60)

    # Chạy thuật toán tự viết
    P_custom, D_custom, P_inv_custom = diagonalize(A_list)

    # --- Trường hợp kỳ vọng trả về None (không chéo hóa được) ---
    if expect_none:
        if P_custom is None:
            print("Kết quả: MA TRẬN KHÔNG CHÉ HÓA ĐƯỢC → Hàm trả về None đúng như kỳ vọng. ĐÚNG")
        else:
            print("Kết quả: Kỳ vọng None nhưng hàm TRẢ VỀ KẾT QUẢ — CẦN KIỂM TRA LẠI. SAI")
        print()
        return

    # --- Trường hợp kỳ vọng chéo hóa được ---
    if P_custom is None:
        print("Kết quả: Hàm trả về None — MA TRẬN KHÔNG CHÉ HÓA ĐƯỢC (hoặc có lỗi). SAI")
        print()
        return

    A  = np.array(A_list, dtype=float)
    P  = np.array(P_custom)
    D  = np.array(D_custom)
    Pi = np.array(P_inv_custom)

    # Kiểm tra 1: A = P * D * P^-1
    A_reconstructed = P @ D @ Pi
    check1 = np.allclose(A, A_reconstructed)
    print("1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A):", "ĐÚNG" if check1 else "SAI")
    if not check1:
        print("   - A gốc:\n", np.round(A, 4))
        print("   - P·D·P⁻¹:\n", np.round(A_reconstructed, 4))

    # Kiểm tra 2: P^-1 * A * P = D (dạng tương đương)
    D_check = Pi @ A @ P
    check2 = np.allclose(D_check, D)
    print("2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp):", "ĐÚNG" if check2 else "SAI")
    if not check2:
        print("   - D (thuật toán):\n", np.round(D, 4))
        print("   - P⁻¹·A·P:\n", np.round(D_check, 4))

    # Kiểm tra 3: P * P^-1 = I
    n = len(A_list)
    I_check = P @ Pi
    check3 = np.allclose(I_check, np.eye(n))
    print("3. Kiểm tra P·P⁻¹ = I:", "ĐÚNG" if check3 else "SAI")

    # Kiểm tra 4: D phải là ma trận đường chéo
    D_off_diag = D - np.diag(np.diag(D))
    check4 = np.allclose(D_off_diag, 0)
    print("4. Kiểm tra D là ma trận đường chéo:", "ĐÚNG" if check4 else "SAI")

    # Đối chiếu trị riêng với NumPy
    eigs_numpy = np.sort(np.linalg.eigvals(A).real)
    eigs_custom = np.sort(np.diag(D))
    check5 = np.allclose(eigs_numpy, eigs_custom, atol=1e-5)
    print("5. So sánh trị riêng với numpy.linalg.eigvals:", "KHỚP" if check5 else "KHÁC NHAU")
    if not check5:
        print(f"   - Tự code:  {np.round(eigs_custom, 4)}")
        print(f"   - NumPy:    {np.round(eigs_numpy,  4)}")

    # In kết quả
    print("\nMa trận P (cột = vector riêng, tự code):\n", np.round(P,  4))
    print("\nMa trận D (trị riêng trên đường chéo):\n",   np.round(D,  4))
    print("\nMa trận P⁻¹ (tự code):\n",                   np.round(Pi, 4))
    print()

## 4.2. **Test Case 1: Ma trận vuông 3×3 — Chéo hóa được thông thường**

In [35]:
A1 = [
    [4, 1, 0],
    [2, 3, 0],
    [0, 0, 5]
]
test_diagonalization("MA TRẬN VUÔNG 3×3 — Trị riêng phân biệt", A1)

KIỂM THỬ: MA TRẬN VUÔNG 3×3 — Trị riêng phân biệt
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 1.  -0.  -0.5]
 [ 1.   0.   1. ]
 [ 0.   1.  -0. ]]

Ma trận D (trị riêng trên đường chéo):
 [[5. 0. 0.]
 [0. 5. 0.]
 [0. 0. 2.]]

Ma trận P⁻¹ (tự code):
 [[ 0.6667  0.3333  0.    ]
 [ 0.      0.      1.    ]
 [-0.6667  0.6667  0.    ]]



## 4.3. **Test Case 2: Ma trận đối xứng 3×3 — Luôn chéo hóa được**

Mục đích: Ma trận đối xứng thực luôn có đủ vector riêng trực giao nhau. Thuật toán phải nhận diện và xử lý đúng.

In [36]:
A2 = [
    [2, 1, 0],
    [1, 3, 1],
    [0, 1, 2]
]
test_diagonalization("MA TRẬN ĐỐI XỨNG 3×3", A2)

KIỂM THỬ: MA TRẬN ĐỐI XỨNG 3×3
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 1. -1.  1.]
 [ 2. -0. -1.]
 [ 1.  1.  1.]]

Ma trận D (trị riêng trên đường chéo):
 [[4. 0. 0.]
 [0. 2. 0.]
 [0. 0. 1.]]

Ma trận P⁻¹ (tự code):
 [[ 0.1667  0.3333  0.1667]
 [-0.5     0.      0.5   ]
 [ 0.3333 -0.3333  0.3333]]



## 4.4. **Test Case 3: Ma trận đường chéo — Đã ở dạng chuẩn sẵn**
Mục đích: Nếu code đúng, P phải là ma trận hoán vị (hoặc ma trận đơn vị), D = A. Kiểm tra edge case đặc biệt.

In [37]:
A3 = [
    [3.0, 0.0, 0.0],
    [0.0, 7.0, 0.0],
    [0.0, 0.0, -2.0]
]
test_diagonalization("MA TRẬN ĐƯỜNG CHÉO (Đã chuẩn sẵn)", A3)

KIỂM THỬ: MA TRẬN ĐƯỜNG CHÉO (Đã chuẩn sẵn)
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 1.  0. -0.]
 [-0.  1. -0.]
 [ 0.  0.  1.]]

Ma trận D (trị riêng trên đường chéo):
 [[ 3.  0.  0.]
 [ 0.  7.  0.]
 [ 0.  0. -2.]]

Ma trận P⁻¹ (tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]



## 4.5. **Test Case 4: Ma trận đơn vị — Trị riêng bội có đủ vector riêng**
Mục đích: Trị riêng λ=1 có bội đại số = 3, và bội hình học = 3 (đủ vector riêng → vẫn chéo hóa được). Thuật toán phải xử lý đúng.

In [38]:
A4 = [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
]
test_diagonalization("MA TRẬN ĐƠN VỊ 3×3 (Trị riêng bội, vẫn chéo hóa được)", A4)

KIỂM THỬ: MA TRẬN ĐƠN VỊ 3×3 (Trị riêng bội, vẫn chéo hóa được)
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Ma trận D (trị riêng trên đường chéo):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Ma trận P⁻¹ (tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]



## 4.6. **Test Case 5: Ma trận 4×4 — Kiểm tra quy mô lớn hơn**

In [39]:
A5 = [
    [1, 0, 0, 0],
    [0, 2, 0, 0],
    [1, 0, 3, 0],
    [0, 1, 0, 4]
]
test_diagonalization("MA TRẬN VUÔNG 4×4 — Trị riêng phân biệt", A5)

KIỂM THỬ: MA TRẬN VUÔNG 4×4 — Trị riêng phân biệt
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 0.  0. -2.  0.]
 [ 0.  0. -0. -2.]
 [ 1.  0.  1. -0.]
 [-0.  1. -0.  1.]]

Ma trận D (trị riêng trên đường chéo):
 [[3. 0. 0. 0.]
 [0. 4. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 2.]]

Ma trận P⁻¹ (tự code):
 [[ 0.5  0.   1.   0. ]
 [ 0.   0.5  0.   1. ]
 [-0.5 -0.  -0.  -0. ]
 [-0.  -0.5 -0.  -0. ]]



## 4.7. **Test Case 6: Ma trận có trị riêng bội — Vẫn chéo hóa được**
Mục đích: Ma trận [[1,0,0],[0,1,0],[0,0,2]] có λ=1 bội 2 nhưng không gian riêng 2 chiều → đủ 3 vector riêng độc lập.

In [40]:
A6 = [
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 2]
]
test_diagonalization("MA TRẬN TRỊ RIÊNG BỘI — λ=1 (bội 2) vẫn chéo hóa được", A6)

KIỂM THỬ: MA TRẬN TRỊ RIÊNG BỘI — λ=1 (bội 2) vẫn chéo hóa được
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 1.  0.  0.]
 [ 0.  1.  0.]
 [-0. -0.  1.]]

Ma trận D (trị riêng trên đường chéo):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 2.]]

Ma trận P⁻¹ (tự code):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]



## 4.8. **Test Case 7: Ma trận KHÔNG chéo hóa được — Jordan Block 2×2**
Mục đích: Ma trận [[2,1],[0,2]] có λ=2 bội đại số = 2 nhưng bội hình học = 1 (thiếu vector riêng). Hàm phải trả về (None, None, None).

In [41]:
A7 = [
    [2.0, 1.0],
    [0.0, 2.0]
]
test_diagonalization("MA TRẬN JORDAN BLOCK 2×2 (KHÔNG CHÉ HÓA ĐƯỢC)", A7, expect_none=True)

KIỂM THỬ: MA TRẬN JORDAN BLOCK 2×2 (KHÔNG CHÉ HÓA ĐƯỢC)
------------------------------------------------------------
Kết quả: MA TRẬN KHÔNG CHÉ HÓA ĐƯỢC → Hàm trả về None đúng như kỳ vọng. ĐÚNG



## 4.9. **Test Case 8: Ma trận KHÔNG chéo hóa được — Jordan Block 3×3**
Mục đích: Ma trận Jordan cấp 3 với λ=5 có bội hình học = 1 nhưng bội đại số = 3 → không đủ vector riêng.

In [42]:
A8 = [
    [5.0, 1.0, 0.0],
    [0.0, 5.0, 1.0],
    [0.0, 0.0, 5.0]
]
test_diagonalization("MA TRẬN JORDAN BLOCK 3×3 (KHÔNG CHÉ HÓA ĐƯỢC)", A8, expect_none=True)

KIỂM THỬ: MA TRẬN JORDAN BLOCK 3×3 (KHÔNG CHÉ HÓA ĐƯỢC)
------------------------------------------------------------
Kết quả: MA TRẬN KHÔNG CHÉ HÓA ĐƯỢC → Hàm trả về None đúng như kỳ vọng. ĐÚNG



## 4.10. **Test Case 9: Ma trận có trị riêng phức — Ngoài phạm vi hỗ trợ**
Mục đích: Ma trận xoay [[0,-1],[1,0]] có trị riêng là ±i (thuần phức). Hàm phải từ chối và trả về (None, None, None).

In [43]:
A9 = [
    [0.0, -1.0],
    [1.0,  0.0]
]
test_diagonalization("MA TRẬN XOA (Trị riêng phức ±i — ngoài hỗ trợ)", A9, expect_none=True)

KIỂM THỬ: MA TRẬN XOA (Trị riêng phức ±i — ngoài hỗ trợ)
------------------------------------------------------------
Kết quả: MA TRẬN KHÔNG CHÉ HÓA ĐƯỢC → Hàm trả về None đúng như kỳ vọng. ĐÚNG



## 4.11. **Test Case 10: Ma trận 2×2 — Kiểm tra cơ bản**

In [44]:
A10 = [
    [3, 1],
    [1, 3]
]
test_diagonalization("MA TRẬN ĐỐI XỨNG 2×2 — Chéo hóa được", A10)

KIỂM THỬ: MA TRẬN ĐỐI XỨNG 2×2 — Chéo hóa được
------------------------------------------------------------
1. Kiểm tra A = P·D·P⁻¹ (tái tạo lại A): ĐÚNG
2. Kiểm tra P⁻¹·A·P = D (chéo hóa trực tiếp): ĐÚNG
3. Kiểm tra P·P⁻¹ = I: ĐÚNG
4. Kiểm tra D là ma trận đường chéo: ĐÚNG
5. So sánh trị riêng với numpy.linalg.eigvals: KHỚP

Ma trận P (cột = vector riêng, tự code):
 [[ 1. -1.]
 [ 1.  1.]]

Ma trận D (trị riêng trên đường chéo):
 [[4. 0.]
 [0. 2.]]

Ma trận P⁻¹ (tự code):
 [[ 0.5  0.5]
 [-0.5  0.5]]

